In [ ]:
# ESTA - H3 feasibility probe, step 2 (GPU kernel). Downloads the GGUF stack and
# generates one clip, instrumented. Writes h3_probe.json + the mp4 to /kaggle/working.
import subprocess, sys, os, json, time, shutil, traceback, urllib.request, urllib.error

OUT = "/kaggle/working"
ROOT = "/kaggle/temp/ComfyUI"
os.makedirs("/kaggle/temp", exist_ok=True)
os.environ["HF_HOME"] = "/kaggle/temp/hf"

WIDTH, HEIGHT, LENGTH, STEPS = (640, 384, 124, 4)
PROMPT = 'a slow cinematic push in on a trading desk at night, three monitors glowing with red and green candlestick charts, shallow depth of field, shot on film, natural light'

report = {"ok": False, "phases": {}, "notes": []}
t0 = time.time()
# Hard wall for the whole kernel. Kaggle's own cap is 12 h and it bills every one
# of those hours against a 30 h/week quota, so an unattended hang is expensive.
# Nothing here should ever take this long; if it does, we want the report, not the wait.
DEADLINE_S = 3000

def check_deadline(where):
    if time.time() - t0 > DEADLINE_S:
        raise TimeoutError(f"global deadline {DEADLINE_S}s exceeded at {where}")

def note(k, v):
    report["notes"].append({k: v}); print(f"[esta] {k}: {v}", flush=True)

def phase(name, start):
    dt = round(time.time() - start, 1)
    report["phases"][name] = dt
    print(f"[esta] PHASE {name}: {dt}s", flush=True)
    return time.time()

def finish():
    report["total_s"] = round(time.time() - t0, 1)
    with open(os.path.join(OUT, "h3_probe.json"), "w") as fh:
        json.dump(report, fh, indent=2)
    print("ESTA_PROBE_RESULT::" + json.dumps(report)[:3000], flush=True)

try:
    import torch, psutil
    note("gpu", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
    note("vram_gb", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
         if torch.cuda.is_available() else 0)
    note("host_ram_gb", round(psutil.virtual_memory().total / 1e9, 1))
    if not torch.cuda.is_available():
        raise RuntimeError("no GPU on this kernel")

    t = time.time()
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/comfyanonymous/ComfyUI", ROOT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    os.path.join(ROOT, "requirements.txt")], check=False)
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/city96/ComfyUI-GGUF",
                    os.path.join(ROOT, "custom_nodes", "ComfyUI-GGUF")], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gguf"], check=False)
    t = phase("install", t)

    # --- weights -------------------------------------------------------------
    # The whole feasibility question in one place: these are the smallest quants
    # that keep the 32B text encoder and the denoiser inside 16 GB VRAM / 32 GB RAM.
    # Auth comes from a Kaggle Secret, never from the notebook source - the source
    # is uploaded to Kaggle and stored there, so a pasted token would leak.
    # Absent secret is fine; it just means the throttled anonymous path.
    try:
        from kaggle_secrets import UserSecretsClient
        _tok = UserSecretsClient().get_secret("HF_TOKEN")
        if _tok:
            os.environ["HF_TOKEN"] = _tok
            os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
            note("hf_auth", "authenticated via kaggle secret")
        else:
            note("hf_auth", "secret empty - anonymous")
    except Exception as e:
        note("hf_auth", f"no kaggle secret ({str(e)[:80]}) - anonymous")

    from huggingface_hub import hf_hub_download
    WANT = [tuple(x) for x in json.loads('[["Abiray/MiniMax-H3-Pruned-GGUF", "MiniMax-H3-FL2VA-Pruned-Q4_K_M.gguf", "unet"], ["Abiray/MiniMax-H3-GGUF", "text_encoders/qwen3vl_32b_minimax_h3-Q4_K_M.gguf", "clip"], ["Comfy-Org/MiniMax-H3", "vae/minimax_h3_video_vae_fp16.safetensors", "vae"], ["lightx2v/Minimax-h3-Turbo", "minimax_h3_fl2v_turbo_4step_v1.1_768p_comfyui_bf16.safetensors", "loras"]]')]
    local = {}
    DL_TIMEOUT_S = 1200
    for repo, fname, kind in WANT:
        check_deadline("download")
        dest_dir = os.path.join(ROOT, "models", kind)
        os.makedirs(dest_dir, exist_ok=True)
        # Downloading in a child process is the only way to bound it: an
        # unauthenticated HF pull that gets rate-limited retries forever inside
        # hf_hub_download, and that is exactly what ate a 12 h kernel once.
        dl_start = time.time()
        code = ("from huggingface_hub import hf_hub_download;"
                f"print(hf_hub_download(repo_id={repo!r}, filename={fname!r}))")
        try:
            r = subprocess.run([sys.executable, "-c", code], capture_output=True,
                               text=True, timeout=DL_TIMEOUT_S)
        except subprocess.TimeoutExpired:
            note("download_timeout", f"{repo}/{fname} exceeded {DL_TIMEOUT_S}s")
            raise TimeoutError(f"download stalled: {fname}")
        if r.returncode != 0:
            note("download_failed", f"{repo}/{fname}: {(r.stderr or '')[-400:]}")
            raise RuntimeError(f"download failed: {fname}")
        p = (r.stdout or "").strip().splitlines()[-1]
        note("download_seconds", f"{os.path.basename(fname)} {round(time.time()-dl_start,1)}s")
        base = os.path.basename(fname)
        link = os.path.join(dest_dir, base)
        if not os.path.exists(link):
            os.symlink(p, link)
        local[kind] = base
        note("downloaded", f"{base} {round(os.path.getsize(p)/1e9,2)}GB")
    t = phase("download", t)

    # --- server --------------------------------------------------------------
    # Tee to disk rather than a pipe nobody reads: ComfyUI's banner states which
    # device it selected, and its per-step output is the only progress signal.
    comfy_log = open(os.path.join(OUT, "comfyui.log"), "w", buffering=1)
    proc = subprocess.Popen(
        [sys.executable, os.path.join(ROOT, "main.py"), "--port", "8188",
         "--disable-auto-launch"],
        cwd=ROOT, stdout=comfy_log, stderr=subprocess.STDOUT, text=True)

    def up():
        try:
            urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=5)
            return True
        except Exception:
            return False

    for i in range(120):
        if up():
            break
        if proc.poll() is not None:
            note("server_exit_code", proc.returncode)
            raise RuntimeError("ComfyUI died on startup")
        time.sleep(2)
    t = phase("server_start", t)

    # Ask the server what it is running on. This is the authoritative answer -
    # the parent process's torch handle says nothing about ComfyUI's choice.
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=15) as r:
            stats = json.loads(r.read().decode())
        devs = [{k: d.get(k) for k in ("name", "type", "vram_total", "vram_free")}
                for d in stats.get("devices", [])]
        note("comfy_devices", devs)
        report["comfy_on_cpu"] = all(
            str(d.get("type", "")).lower() == "cpu" for d in devs) if devs else None
    except Exception as e:
        note("system_stats_failed", str(e)[:200])

    # --- graph ---------------------------------------------------------------
    # Authored from the step-1 /object_info dump, not guessed. MiniMaxH3ImageToVideo
    # emits BOTH the positive conditioning and the AV latent, and doubles as the
    # t2v node when first_frame is omitted.
    G = {
        "1": {"class_type": ("UnetLoaderGGUF" if local["unet"].endswith(".gguf")
                             else "UNETLoader"),
              "inputs": ({"unet_name": local["unet"]} if local["unet"].endswith(".gguf")
                         else {"unet_name": local["unet"], "weight_dtype": "default"})},
        "2": {"class_type": "MiniMaxH3SigmaShift",
              "inputs": {"model": ["1", 0], "shift_video": 12.0, "shift_audio": 3.0}},
        "3": {"class_type": "LoraLoaderModelOnly",
              "inputs": {"model": ["2", 0], "lora_name": local["loras"],
                         "strength_model": 1.0}},
        "4": {"class_type": ("CLIPLoaderGGUF" if local["clip"].endswith(".gguf")
                             else "CLIPLoader"),
              "inputs": {"clip_name": local["clip"], "type": "minimax"}},
        "5": {"class_type": "VAELoader", "inputs": {"vae_name": local["vae"]}},
        "6": {"class_type": "MiniMaxH3ImageToVideo",
              "inputs": {"clip": ["4", 0], "vae": ["5", 0], "prompt": PROMPT,
                         "width": WIDTH, "height": HEIGHT, "length": LENGTH}},
        "7": {"class_type": "ConditioningZeroOut", "inputs": {"conditioning": ["6", 0]}},
        "8": {"class_type": "KSampler",
              "inputs": {"model": ["3", 0], "seed": 12345, "steps": STEPS, "cfg": 1.0,
                         "sampler_name": "euler", "scheduler": "simple",
                         "positive": ["6", 0], "negative": ["7", 0],
                         "latent_image": ["6", 1], "denoise": 1.0}},
        "9": {"class_type": "VAEDecode", "inputs": {"samples": ["8", 0], "vae": ["5", 0]}},
        "10": {"class_type": "CreateVideo", "inputs": {"images": ["9", 0], "fps": 24}},
        "11": {"class_type": "SaveVideo",
               "inputs": {"video": ["10", 0], "filename_prefix": "h3_probe",
                          "format": "mp4", "codec": "h264"}},
    }
    with open(os.path.join(OUT, "graph_sent.json"), "w") as fh:
        json.dump(G, fh, indent=2)

    body = json.dumps({"prompt": G, "client_id": "esta"}).encode()
    req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=body,
                                 headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            resp = json.loads(r.read().decode())
    except urllib.error.HTTPError as e:
        detail = e.read().decode("utf-8", "replace")[:4000]
        note("prompt_rejected", detail)
        with open(os.path.join(OUT, "prompt_error.json"), "w") as fh:
            fh.write(detail)
        raise RuntimeError("ComfyUI rejected the graph - see prompt_error.json")
    pid = resp.get("prompt_id")
    note("prompt_id", pid)

    # --- wait ----------------------------------------------------------------
    hist, waited = None, 0
    while waited < 5400 and time.time() - t0 < DEADLINE_S:
        time.sleep(10); waited += 10
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:8188/history/{pid}", timeout=15) as r:
                h = json.loads(r.read().decode())
            if pid in h:
                hist = h[pid]
                break
        except Exception:
            pass
        if waited % 120 == 0:
            print(f"[esta] still sampling, {waited}s", flush=True)
    t = phase("generate", t)

    if hist is None:
        note("timeout", "no history after 90 min")
    else:
        status = (hist.get("status") or {})
        note("run_status", status.get("status_str"))
        if status.get("status_str") != "success":
            with open(os.path.join(OUT, "run_error.json"), "w") as fh:
                json.dump(hist, fh, indent=2)
            note("run_messages", json.dumps(status.get("messages", []))[:1500])

    for root, _, files in os.walk(os.path.join(ROOT, "output")):
        for f in files:
            if f.endswith((".mp4", ".webm", ".png")):
                shutil.copy(os.path.join(root, f), os.path.join(OUT, f))
                note("artifact", f"{f} {round(os.path.getsize(os.path.join(root,f))/1e6,2)}MB")
                report["ok"] = True

    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=15) as r:
            end_stats = json.loads(r.read().decode())
        note("devices_after_run", [
            {k: d.get(k) for k in ("name", "type", "vram_total", "vram_free")}
            for d in end_stats.get("devices", [])])
    except Exception as e:
        note("end_stats_failed", str(e)[:200])
    note("host_ram_used_gb", round(psutil.virtual_memory().used / 1e9, 2))
    try:
        comfy_log.flush()
        tail = open(os.path.join(OUT, "comfyui.log"), errors="replace").read()[-4000:]
        note("comfy_log_tail", tail[-1500:])
    except Exception:
        pass
    try:
        proc.kill()
    except Exception:
        pass
except Exception as e:
    note("fatal", str(e)[:400])
    traceback.print_exc()
finish()
